In [1]:
import gc, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
warnings.filterwarnings("ignore")

BASE_DIR      = Path("/Users/arjun/Documents/UAV thesis/uav-thesis/uav_px4_dataset_project")
FOLDER_VER    = "v3_voltage_random10_issue_flight-removed"
FILE_VER      = "v4_voltage_random10_cleaned"
DATA_DIR      = BASE_DIR / f"model_ready_no_dr_{FOLDER_VER}"
OUTPUT_DIR    = BASE_DIR / "model_outputs" / f"lstm_ae_anomaly_score_{FOLDER_VER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_FEATURES = [
    "battery_status__cell_count", "battery_status__current_a",
    "battery_status__current_filtered_a", "battery_status__discharged_mah",
    "battery_status__remaining", "battery_status__scale",
    "battery_status__voltage_filtered_v", "battery_status__voltage_v",
    "cumulative_energy_wh", "power_w", "time_s",
    "voltage_per_cell", "voltage_filtered_per_cell",
]

SPLITS = [
    ("split_01_seed_11",  11), ("split_02_seed_22",  22),
    ("split_03_seed_33",  33), ("split_04_seed_44",  44),
    ("split_05_seed_55",  55), ("split_06_seed_66",  66),
    ("split_07_seed_77",  77), ("split_08_seed_88",  88),
    ("split_09_seed_99",  99), ("split_10_seed_111", 111),
]

SEQ_LEN    = 30
BATCH_SIZE = 512
EPOCHS     = 50
PATIENCE   = 10
HIDDEN     = 64
LATENT     = 16

device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: mps


In [2]:
class LSTMAutoEncoder(nn.Module):
    def __init__(self, input_size, hidden=64, latent=16):
        super().__init__()
        # Encoder
        self.encoder_lstm = nn.LSTM(input_size, hidden, batch_first=True)
        self.encoder_fc   = nn.Linear(hidden, latent)

        # Decoder init projections
        self.decoder_input_fc = nn.Linear(latent, input_size)
        self.decoder_h0_fc    = nn.Linear(latent, hidden)
        self.decoder_c0_fc    = nn.Linear(latent, hidden)

        # Decoder
        self.decoder_lstm = nn.LSTM(input_size, hidden, batch_first=True)
        self.output_layer = nn.Linear(hidden, input_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        _, (h, c)  = self.encoder_lstm(x)
        z          = self.encoder_fc(h[-1])           # (batch, latent)

        # Decoder — reverse sequence
        dec_input  = self.decoder_input_fc(z)         # (batch, input_size)
        dec_input  = dec_input.unsqueeze(1).repeat(1, x.size(1), 1)  # (batch, seq, input_size)
        h0 = self.decoder_h0_fc(z).unsqueeze(0)      # (1, batch, hidden)
        c0 = self.decoder_c0_fc(z).unsqueeze(0)
        out, _     = self.decoder_lstm(dec_input, (h0, c0))
        recon      = self.output_layer(out)            # (batch, seq, input_size)
        return recon

print("Model defined ✓")

Model defined ✓


In [3]:
class FlightSeqDataset(torch.utils.data.Dataset):
    def __init__(self, df, feat_cols, seq_len):
        seqs = []
        for _, grp in df.sort_values(["log_id","time_s"]).groupby("log_id", sort=False):
            X = grp[feat_cols].to_numpy(dtype=np.float32)
            n = len(X)
            if n <= seq_len:
                continue
            for i in range(seq_len, n):
                seqs.append(X[i-seq_len:i])
        self.X = torch.from_numpy(np.array(seqs, dtype=np.float32))

    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i]

print("Dataset defined")

Dataset defined


In [5]:
def train_lstm_ae(model, train_ds, val_ds, ckpt_path, seed):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5)

    g = torch.Generator(); g.manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, generator=g)
    val_loader = torch.utils.data.DataLoader(
        val_ds, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

    best_val, pat_ctr = np.inf, 0
    for epoch in range(1, EPOCHS+1):
        model.train()
        tl = []
        for xb in train_loader:
            xb = xb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), xb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl.append(loss.item())

        model.eval()
        vl = []
        with torch.no_grad():
            for xb in val_loader:
                vl.append(criterion(model(xb.to(device)), xb.to(device)).item())
        v_loss = np.mean(vl)
        scheduler.step(v_loss)

        if v_loss < best_val - 1e-6:
            best_val, pat_ctr = v_loss, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            pat_ctr += 1
            if pat_ctr >= PATIENCE:
                print(f"    Early stop ep {epoch}  best_val={best_val:.6f}")
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    model.eval()
    return model


def score_test(model, test_df, feat_cols, seq_len):
    """Per-timestep anomaly score = mean reconstruction error over sequence."""
    model.eval()
    # accumulate scores per position
    score_sum   = np.zeros(len(test_df), dtype=np.float64)
    score_count = np.zeros(len(test_df), dtype=np.int32)

    test_sorted = test_df.sort_values(["log_id","time_s"]).reset_index()
    orig_idx    = test_sorted["index"].values   # maps back to test_df index

    offset = 0
    for lid, grp in test_sorted.groupby("log_id", sort=False):
        X = grp[feat_cols].to_numpy(dtype=np.float32)
        n = len(X)
        if n <= seq_len:
            offset += n
            continue
        seqs = np.stack([X[i-seq_len:i] for i in range(seq_len, n)])
        with torch.no_grad():
            for bi, batch in enumerate(torch.split(torch.tensor(seqs), BATCH_SIZE)):
                recon = model(batch.to(device)).cpu().numpy()
                errs  = ((seqs[bi*BATCH_SIZE:(bi+1)*BATCH_SIZE] - recon)**2).mean(axis=(1,2))
                for j, err in enumerate(errs):
                    pos = offset + seq_len + bi*BATCH_SIZE + j
                    score_sum[pos]   += err
                    score_count[pos] += 1
        offset += n

    # rows with count=0 (first seq_len rows per flight) get NaN then ffill
    scores = np.where(score_count > 0, score_sum / score_count, np.nan)
    result = test_sorted.copy()
    result["anomaly_score_lstm_ae"] = scores
    result["anomaly_score_lstm_ae"] = result.groupby("log_id")["anomaly_score_lstm_ae"].transform(
        lambda x: x.ffill().bfill())
    return result[["log_id","time_s","anomaly_score_lstm_ae","dataset_label"]]


# ── Main loop ─────────────────────────────────────────────────
for split_name, seed in SPLITS:
    print(f"\n{'='*60}")
    print(f"SPLIT: {split_name}  SEED: {seed}")
    print(f"{'='*60}")

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.backends.mps.is_available(): torch.mps.manual_seed(seed)

    split_dir  = DATA_DIR / split_name
    out_dir    = OUTPUT_DIR / split_name
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file   = out_dir / f"test_lstm_ae_scored_{split_name}.parquet"

    if out_file.exists():
        print("  Already scored — skipping")
        continue

    train_df = pd.read_parquet(split_dir / f"train_no_dr_scaled_{FILE_VER}_{split_name}.parquet")
    val_df   = pd.read_parquet(split_dir / f"val_no_dr_scaled_{FILE_VER}_{split_name}.parquet")
    test_df  = pd.read_parquet(split_dir / f"test_no_dr_scaled_{FILE_VER}_{split_name}.parquet")

    available = [f for f in BASE_FEATURES if f in train_df.columns]
    print(f"  Features: {len(available)}")

    for df in [train_df, val_df, test_df]:
        for col in df.select_dtypes("float64").columns:
            df[col] = df[col].astype(np.float32)

    train_ds = FlightSeqDataset(train_df, available, SEQ_LEN)
    val_ds   = FlightSeqDataset(val_df,   available, SEQ_LEN)
    print(f"  Train seqs: {len(train_ds):,}  Val seqs: {len(val_ds):,}")

    ckpt_path = out_dir / f"ckpt_lstm_ae_{split_name}.pt"
    model = LSTMAutoEncoder(input_size=len(available), hidden=HIDDEN, latent=LATENT).to(device)

    if ckpt_path.exists():
        print("  Checkpoint exists — loading")
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
        model.eval()
    else:
        print("  Training...")
        model = train_lstm_ae(model, train_ds, val_ds, ckpt_path, seed)

    print("  Scoring test set...")
    scored_df = score_test(model, test_df, available, SEQ_LEN)
    scored_df.to_parquet(out_file, index=False)
    print(f"  Saved: {out_file.name}")
    print(f"  Score stats — mean:{scored_df['anomaly_score_lstm_ae'].mean():.6f}  "
          f"max:{scored_df['anomaly_score_lstm_ae'].max():.6f}")

    del model, train_ds, val_ds, train_df, val_df, test_df; gc.collect()
    if device.type == "mps": torch.mps.empty_cache()

print("\n All splits done")


SPLIT: split_01_seed_11  SEED: 11
  Features: 13
  Train seqs: 695,513  Val seqs: 132,527
  Checkpoint exists — loading
  Scoring test set...
  Saved: test_lstm_ae_scored_split_01_seed_11.parquet
  Score stats — mean:0.011001  max:0.692530

SPLIT: split_02_seed_22  SEED: 22
  Features: 13
  Train seqs: 705,382  Val seqs: 121,126
  Training...
    Early stop ep 33  best_val=0.003991
  Scoring test set...
  Saved: test_lstm_ae_scored_split_02_seed_22.parquet
  Score stats — mean:0.006552  max:0.212213

SPLIT: split_03_seed_33  SEED: 33
  Features: 13
  Train seqs: 659,840  Val seqs: 132,834
  Training...
  Scoring test set...
  Saved: test_lstm_ae_scored_split_03_seed_33.parquet
  Score stats — mean:0.002958  max:0.076312

SPLIT: split_04_seed_44  SEED: 44
  Features: 13
  Train seqs: 674,482  Val seqs: 128,455
  Training...
    Early stop ep 46  best_val=0.010119
  Scoring test set...
  Saved: test_lstm_ae_scored_split_04_seed_44.parquet
  Score stats — mean:0.003700  max:0.219986

SPL